## Fitting a Vol Surface with Rough Bergomi

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import datetime
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, TensorDataset
from tqdm.notebook import tqdm, trange
from scipy.optimize import minimize
import pandas as pd
import QuantLib as ql
from py_vollib.black_scholes.implied_volatility import implied_volatility
from mpl_toolkits.mplot3d import Axes3D
from scipy.interpolate import griddata
from joblib import load
import math

In [ ]:
if torch.cuda.is_available():
    device = torch.device("cuda:0")
else:
    device = torch.device("cpu")
print("Device:", device)

### Load and prepare the vol surface

In [ ]:
in_file = 'SPXOptions.csv'
dfopt = pd.read_csv(in_file)

In [ ]:
dfopt.columns

In [ ]:
dfopt['quote_date'].unique()

In [ ]:
dfvol = dfopt[dfopt['quote_date'] == '2013-03-27']
dfvol = dfvol[dfvol['type'] == 'put']
underlying = dfvol['underlying'].unique()

In [ ]:
drop_list = ['contract', 'underlying', 'expiration', 'type', 'style',
       'bid', 'ask', 'volume', 'open_interest', 'quote_date', 'delta', 'gamma',
       'theta', 'vega', 'rate', '20D_HV','40D_HV', '60D_HV', 'BlackScholes', 'mid', 'error']
dfvol.drop(drop_list, axis=1, inplace=True)

In [ ]:
dfvol.head()

In [ ]:
dfvol['relative_strike'] = dfvol['strike'] / underlying.item()

In [ ]:
dfvol.drop(['strike'], axis=1, inplace=True)

In [ ]:
dfvol = dfvol[dfvol['relative_strike'] < 1.5]
dfvol = dfvol[dfvol['relative_strike'] > 0.5]
dfvol = dfvol[dfvol['maturity'] > 0.4]

In [ ]:
dfvol['maturity'].unique()

In [ ]:
class rbergomiiv:
    def __init__(self, model, sc, device):
        self.model = model
        self.sc = sc
        self.device = device
    
    def impliedvol(self, eta, rho, H, xi_0, mats, strikes):
        no_opt = len(mats)
        veta = np.full(no_opt, eta)
        vrho = np.full(no_opt, rho)
        vH = np.full(no_opt, H)
        vxi_0 = np.full(no_opt, xi_0)
        param_dict = {'eta': veta, 'rho': vrho,
                      'H': vH, 'xi_0': vxi_0,
                      'tau': mats, 'K': strikes}       
        df = pd.DataFrame(param_dict)
        bergomi_X = self.sc.transform(df)
        tbergomi_X = torch.Tensor(bergomi_X).to(self.device)
        bergominn_iv = self.model(tbergomi_X)
        return bergominn_iv.squeeze().detach().cpu().numpy()  

In [ ]:
def objective_function(params, iv_surface, rbergomiobj):
    print(params)
    total_error = 0
    eta = params[0]
    rho = params[1]
    H = params[2]
    xi_0 = params[3]
    mats = iv_surface['maturity']
    strikes = iv_surface['relative_strike']
    model_iv = rbergomiobj.impliedvol(eta, rho, H, xi_0, mats, strikes)
    market_iv = iv_surface['implied_volatility'].to_numpy()
    mse = np.mean((market_iv - model_iv) ** 2)
    print(mse)
    return mse

In [ ]:
S0 = 1.0
eta = 2.5
rho = -0.5
H = 0.25
xi_0 = 0.075
param_lst = [eta, rho, H, xi_0]
initial_params = np.array(param_lst)

In [ ]:
class Swish(nn.Module):
    def __init__(self):
        super(Swish, self).__init__()

    def forward(self, x):
        return x * torch.sigmoid(x)

class NeuralNetwork(nn.Module):
    def __init__(self, width = 32, use_batch_norm=False):
        super(NeuralNetwork, self).__init__()
        self.use_batch_norm = use_batch_norm
        self.activation = Swish()
        self.fc1 = nn.Linear(6, width)
        self.bn1 = nn.BatchNorm1d(width) if use_batch_norm else nn.Identity()
        self.fc2 = nn.Linear(width, width)
        self.bn2 = nn.BatchNorm1d(width) if use_batch_norm else nn.Identity()
        self.fc3 = nn.Linear(width, width)
        self.bn3 = nn.BatchNorm1d(width) if use_batch_norm else nn.Identity()
        self.fc4 = nn.Linear(width, width)
        self.bn4 = nn.BatchNorm1d(width) if use_batch_norm else nn.Identity()
        self.fc5 = nn.Linear(width, 1)

    def forward(self, x):
        x = self.activation(self.bn1(self.fc1(x)))
        x = self.activation(self.bn2(self.fc2(x)))
        x = self.activation(self.bn3(self.fc3(x)))
        x = self.activation(self.bn4(self.fc4(x)))
        x = self.fc5(x)
        return x

In [ ]:
bergominn = NeuralNetwork(256, True).to(device)

In [ ]:
model_path = "bergominn.pth"
state_dict = torch.load(model_path)

In [ ]:
bergominn.load_state_dict(state_dict)

In [ ]:
scaler_path = 'rbergomi_scaler.bin'
standard_scaler = load(scaler_path)

In [ ]:
bergomiivobj = rbergomiiv(bergominn, standard_scaler, device)

In [ ]:
tmats = [1.0, 1.0]
tstrikes = [1.0, 1.0]

In [ ]:
bergomiivobj.impliedvol(eta, rho, H, xi_0, tmats, tstrikes)

In [ ]:
optimization_method = 'Nelder-Mead'

In [ ]:
bounds = [[0.5, 4.0], [-0.95, -0.1], [0.025, 0.5], [0.01, 0.16]]

In [ ]:
result = minimize(objective_function, initial_params, args=(dfvol,bergomiivobj), 
                  bounds=bounds, method=optimization_method, tol=1e-10)

In [ ]:
if result.success:
    fitted_params = result.x
    print("Optimization succeeded.")
    print(f"Fitted rBergomi parameters: {fitted_params}")
else:
    print("Optimization failed.")
    print(result.message)

In [ ]:
feta, frho, fH, fxi_0 = result.x

In [ ]:
rbiv = bergomiivobj.impliedvol(feta, frho, fH, fxi_0, dfvol['maturity'], dfvol['relative_strike'])

In [ ]:
model_ivd = {'implied_volatility': rbiv, 'maturity': dfvol['maturity'].to_numpy(), 
             'relative_strike': dfvol['relative_strike'].to_numpy()}

In [ ]:
options = pd.DataFrame(model_ivd)

In [ ]:
from py_vollib.black_scholes.implied_volatility import implied_volatility
import rbergomi

def rBergomiIV(S0, K, T, rho, eta, H, xi_0, num_paths, num_steps):
    a = H - 0.5
    rB = rbergomi.rBergomi(n = num_steps, N = num_paths, T = T, a = a)
    dW1 = rB.dW1()
    dW2 = rB.dW2()
    Y = rB.Y(dW1)
    dB = rB.dB(dW1, dW2, rho = rho)
    V = rB.V(Y, xi = xi_0, eta = eta)
    S = rB.S(V, dB)
    ST = S[:,-1][:,np.newaxis]
    call_payoff = np.maximum(ST - K,0)
    call_price = np.mean(call_payoff, axis = 0)[:,np.newaxis].item()
    iv = implied_volatility(call_price, S0, K, T, 0.0, 'c')
    return iv

In [ ]:
dfrbermgomiform = pd.DataFrame()
for imat in dfvol['maturity']:
    for istrike in dfvol['relative_strike']:
        try:
            iv = rBergomiIV(S0, istrike, imat, frho, feta, fH, fxi_0, 30000, 100)
            new_row = {'implied_volatility': iv, 'maturity': imat, 
             'relative_strike': istrike}
            dfrbermgomiform.append(new_row, ignore_index=True)
        except:
            continue

In [ ]:
dfrbermgomiform

In [ ]:
unique_maturities = pd.concat([dfvol['maturity'], model_dfvol['maturity']]).unique()
unique_maturities.sort()  # Sort maturities for a more logical plot arrangement

# Determine the grid size for subplots
n = len(unique_maturities)
cols = 3  # You can adjust this value based on your preference
rows = math.ceil(n / cols)

# Step 2: Create subplots for each maturity
plt.figure(figsize=(20, 6 * rows))
for i, maturity in enumerate(unique_maturities, start=1):
    plt.subplot(rows, cols, i)
    
    # Filter data for the current maturity
    market_subset = dfvol[dfvol['maturity'] == maturity]
    model_subset = model_dfvol[model_dfvol['maturity'] == maturity]
    form_subset = dfrbermgomiform[dfrbermgomiform['maturity'] == maturity]

    # Make sure data is sorted by relative_strike for smooth lines
    market_subset = market_subset.sort_values('relative_strike')
    model_subset = model_subset.sort_values('relative_strike')
    form_subset = form_subset.sort_values('relative_strike')

    # Plotting
    plt.plot(market_subset['relative_strike'], market_subset['implied_volatility'], label='Market IV', linestyle='-')
    plt.plot(model_subset['relative_strike'], model_subset['implied_volatility'], label='Model IV', linestyle='--')
    plt.plot(form_subset['relative_strike'], form_subset['implied_volatility'], label='Form IV', linestyle='-.')

    plt.title(f'Maturity {maturity}')
    plt.xlabel('Relative Strike')
    plt.ylabel('Implied Volatility')
    plt.legend()
    plt.grid(True)

plt.tight_layout()
plt.show()